# Bayesian Feature Selection Demo (MCMC + Horseshoe Prior)

This notebook demonstrates Bayesian feature selection with the **regularized horseshoe prior** using the tools in this repository.

## Learning goals
1. Understand the math behind horseshoe shrinkage and sparsity.
2. Run MCMC inference with `HorseshoeGLM` on simulated sparse data.
3. Evaluate convergence and posterior behavior with **ArviZ**.
4. Inspect posterior betas and feature selection quality.
5. See a GPU-ready workflow template for real datasets.
6. Practice with hands-on exercises that reveal strengths and limits of the method.


## 1) Mathematical intuition: why the horseshoe induces sparsity

For feature coefficients \(\beta_j\), the (regularized) horseshoe hierarchy is:

\[
\tau \sim \text{Half-Cauchy}(0, \tau_0), \quad
\lambda_j \sim \text{Half-Cauchy}(0, 1), \quad
c^2 \sim \text{Inverse-Gamma}(1,1)
\]
\[
\tilde{\lambda}_j = \sqrt{\frac{c^2\lambda_j^2}{c^2 + \tau^2\lambda_j^2}}, \quad
\beta_j \mid \tau,\lambda_j,c^2 \sim \mathcal{N}(0,\tau^2\tilde{\lambda}_j^2)
\]

For a Gaussian GLM:
\[
\eta_i = \alpha + x_i^\top\beta, \quad
y_i \sim \mathcal{N}(\eta_i, \sigma^2)
\]

### Key sparsity mechanism
- **Global shrinkage** \(\tau\): pushes all coefficients toward zero.
- **Local shrinkage** \(\lambda_j\): lets a few relevant features escape shrinkage.
- **Slab regularization** \(c^2\): avoids unrealistically huge coefficients.

A useful view is the shrinkage factor \(\kappa_j = 1/(1 + \tau^2\lambda_j^2)\):
- Noise features often have \(\kappa_j \approx 1\) (strong shrinkage to near zero).
- True signals often have smaller \(\kappa_j\), preserving non-zero effects.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import jax
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

from bayesian_feature_selection import HorseshoeGLM, InferenceConfig

np.random.seed(42)
print('JAX devices:', jax.devices())


## 2) Simulated sparse data

We create a high-dimensional-ish problem with many irrelevant features and a few true signals.


In [ ]:
# Simulated data with sparse truth
n, p = 220, 60
n_signal = 6
signal_idx = np.array([0, 3, 7, 13, 27, 42])

X = np.random.normal(size=(n, p))
true_beta = np.zeros(p)
true_beta[signal_idx] = np.array([2.8, -2.2, 1.8, -1.6, 1.2, 0.9])

noise_sd = 1.0
y = X @ true_beta + np.random.normal(0, noise_sd, size=n)

# standardize features for stable inference
X = StandardScaler().fit_transform(X)

print(f'n={n}, p={p}, true non-zero features={len(signal_idx)}')

plt.figure(figsize=(10, 3.5))
plt.stem(np.arange(p), true_beta, basefmt='k-')
plt.title('Ground-truth coefficients (sparse)')
plt.xlabel('Feature index')
plt.ylabel('True beta')
plt.grid(alpha=0.3)
plt.show()


## 3) Fit Horseshoe GLM with MCMC

We focus on MCMC inference (`method='mcmc'`) for full posterior uncertainty.

Rule-of-thumb initialization for global scale:
\[
\tau_0 \approx \frac{p_0}{p-p_0}\cdot\frac{1}{\sqrt{n}}
\]
where \(p_0\) is expected number of relevant features.


In [ ]:
expected_p0 = n_signal
scale_global = (expected_p0 / max(p - expected_p0, 1)) / np.sqrt(n)
scale_global = float(np.clip(scale_global, 0.05, 1.0))

use_gpu = any(d.platform == 'gpu' for d in jax.devices())
print('Using GPU?', use_gpu)
print('scale_global:', round(scale_global, 4))

model = HorseshoeGLM(family='gaussian', scale_global=scale_global)
config = InferenceConfig(
    method='mcmc',
    num_warmup=300,
    num_samples=500,
    num_chains=2,
    use_gpu=use_gpu,
    progress_bar=False,
)

model.fit(X, y, config=config)
print('MCMC complete.')


## 4) Feature selection results

We use posterior inclusion from beta samples (default behavior in this repo) and compare with ground truth.


In [ ]:
importance = model.get_feature_importance(method='beta', threshold=0.5)
importance = importance.sort_values('beta_inclusion_prob', ascending=False).reset_index(drop=True)

selected_idx = importance.loc[importance['selected'], 'feature_idx'].to_numpy()
true_mask = np.isin(np.arange(p), signal_idx).astype(int)
pred_mask = np.isin(np.arange(p), selected_idx).astype(int)

precision = precision_score(true_mask, pred_mask, zero_division=0)
recall = recall_score(true_mask, pred_mask, zero_division=0)
f1 = f1_score(true_mask, pred_mask, zero_division=0)

print(f'Selected features: {len(selected_idx)}')
print(f'Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}')

show_cols = ['feature_idx', 'beta_mean', 'beta_lower_95', 'beta_upper_95', 'beta_inclusion_prob', 'selected']
display(importance[show_cols].head(12))


In [ ]:
# Visual comparison: posterior mean vs truth
posterior_mean = np.zeros(p)
posterior_mean[importance['feature_idx'].to_numpy()] = importance['beta_mean'].to_numpy()

plt.figure(figsize=(11, 4))
plt.plot(true_beta, 'o-', label='true beta', alpha=0.8)
plt.plot(posterior_mean, 's--', label='posterior mean beta', alpha=0.8)
plt.axhline(0, color='k', lw=1)
plt.title('True vs posterior-mean coefficients')
plt.xlabel('Feature index')
plt.ylabel('Coefficient value')
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 5) ArviZ diagnostics: convergence + beta visualization

We convert NumPyro output to `InferenceData`, inspect R-hat/ESS, and visualize chains/posteriors.


In [ ]:
idata = az.from_numpyro(model.mcmc)

# Summaries for convergence diagnostics
summary = az.summary(idata, var_names=['tau', 'c2', 'beta'])
summary.head()


In [ ]:
# Basic convergence checks
rhat_vals = pd.to_numeric(summary['r_hat'], errors='coerce').dropna()
ess_vals = pd.to_numeric(summary['ess_bulk'], errors='coerce').dropna()

rhat_max = float(rhat_vals.max()) if len(rhat_vals) else float('nan')
ess_min = float(ess_vals.min()) if len(ess_vals) else float('nan')

print(f'max R-hat: {rhat_max:.4f}')
print(f'min ESS (bulk): {ess_min:.1f}')
print('Convergence guideline: R-hat near 1.00 and reasonably large ESS.')


In [ ]:
# Trace plot for global parameters
az.plot_trace(idata, var_names=['tau', 'c2'])
plt.tight_layout()
plt.show()


In [ ]:
# Beta visualization with ArviZ for top features by inclusion probability
beta_dim = [c for c in idata.posterior.coords if c.startswith('beta_dim_')][0]
top_k = 8
top_features = importance['feature_idx'].head(top_k).tolist()

az.plot_trace(
    idata,
    var_names=['beta'],
    coords={beta_dim: top_features},
)
plt.tight_layout()
plt.show()


## 6) Real data + GPU-ready workflow (optional run)

Below is a compact template using a real dataset (`sklearn` breast cancer data) and automatic GPU detection.

- Set `RUN_REAL_DATA_DEMO = True` to execute.
- Increase MCMC samples/chains for production analysis.


In [ ]:
RUN_REAL_DATA_DEMO = False

if RUN_REAL_DATA_DEMO:
    ds = load_breast_cancer()
    X_real = StandardScaler().fit_transform(ds.data)
    y_real = ds.target.astype(float)

    use_gpu_real = any(d.platform == 'gpu' for d in jax.devices())
    model_real = HorseshoeGLM(family='binomial', scale_global=0.25)

    config_real = InferenceConfig(
        method='mcmc',
        num_warmup=200,
        num_samples=300,
        num_chains=2,
        use_gpu=use_gpu_real,
        progress_bar=False,
    )

    model_real.fit(X_real, y_real, config=config_real)
    idata_real = az.from_numpyro(model_real.mcmc)

    print('Real-data model fit complete. Device GPU?', use_gpu_real)
    display(az.summary(idata_real, var_names=['tau', 'c2']).head())
else:
    print('Skipped. Set RUN_REAL_DATA_DEMO = True to run the real-data section.')


## 7) Hands-on exercises and challenge prompts

Try these short activities to build intuition and discover practical tradeoffs.

### Exercise A — Sparsity calibration (global shrinkage)
1. Sweep `scale_global` over `[0.05, 0.1, 0.2, 0.5, 1.0]`.
2. Track precision/recall/F1 of selected features.
3. Explain the bias-variance tradeoff you observe.

### Exercise B — Signal-to-noise stress test
1. Keep the same true coefficients, vary `noise_sd` from `0.3` to `2.5`.
2. Plot recovery metrics vs noise level.
3. Identify where horseshoe starts missing weak signals.

### Exercise C — Correlated predictors (method limitation)
1. Create blocks of strongly correlated features.
2. Place a true signal in one feature per block.
3. Compare selection under `method='beta'`, `'lambda'`, and `'both'`.
4. Discuss why posterior mass can spread across correlated variables.

### Exercise D — p >> n scenario (state-of-the-art use case)
1. Set `n=80`, `p=500`, true signals = 8.
2. Use MCMC with fewer warmup/samples first, then increase.
3. Compare runtime, convergence, and recovery quality.

### Exercise E — Chain quality diagnostics
1. Intentionally set very short warmup/samples.
2. Use ArviZ to inspect R-hat, ESS, and trace behavior.
3. Propose a principled tuning plan to improve inference.

### Exercise F — Real-data challenge
1. Run the optional breast-cancer section with `RUN_REAL_DATA_DEMO=True`.
2. Rank features by inclusion probability.
3. Compare selected features to coefficients from a lasso baseline.
4. Reflect: when does Bayesian uncertainty add value over point-estimate methods?

### Exercise G — Prior robustness challenge
1. Replace true coefficients with one very large signal + several weak ones.
2. Evaluate whether regularized horseshoe retains weak effects while controlling noise.
3. Summarize pros/cons for scientific discovery vs pure prediction tasks.


## 8) Takeaways

- The horseshoe prior combines **aggressive noise shrinkage** with **heavy-tail signal retention**.
- MCMC gives richer uncertainty quantification for feature relevance than point-estimate methods.
- ArviZ diagnostics are essential before trusting selected features.
- Performance depends on prior scale, SNR, and predictor correlation structure.
